In [6]:
#!pip install gliner


In [3]:
from gliner import GLiNER
from sklearn.metrics import classification_report

def read_conll_file(filepath):
    tokens = []
    labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if tokens:
                    yield tokens, labels
                    tokens, labels = [], []
            else:
                splits = line.split()
                tokens.append(splits[0])
                labels.append(splits[1])
        if tokens:
            yield tokens, labels

def bio_to_entities(tokens, bio_labels):
    entities = []
    entity = None
    for i, tag in enumerate(bio_labels):
        if tag.startswith("B-"):
            if entity:
                entities.append(entity)
            entity = {"label": tag[2:], "tokens": [tokens[i]], "start": i, "end": i}
        elif tag.startswith("I-") and entity and tag[2:] == entity["label"]:
            entity["tokens"].append(tokens[i])
            entity["end"] = i
        else:
            if entity:
                entities.append(entity)
                entity = None
    if entity:
        entities.append(entity)
    return entities

def entity_set_from_bio(tokens, bio_labels):
    ents = bio_to_entities(tokens, bio_labels)
    # Return set of (text, label)
    return set((" ".join(e["tokens"]), e["label"]) for e in ents)

def entity_set_from_pred(pred_entities):
    # pred_entities already has "text" and "label"
    return set((e["text"], e["label"]) for e in pred_entities)

# Load model
model = GLiNER.from_pretrained("urchade/gliner_largev2")



Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [15]:
conll_filepath = "synthetic_test_v1.conll"

all_true_labels = []
all_pred_labels = []

# Entity list for prediction
labels_list = [
    "EquipmentName",
    "DeviceProperties",
    "FaultCode",
    "FaultType",
    "MaintenanceMethod",
    "VehicleComponentLocation",
    "MeasurementValue",
    "VehicleMakeModel",
    "TimeDuration",
    "CustomerReportedSymptom"
]

for tokens, labels in read_conll_file(conll_filepath):
    text = " ".join(tokens)
    pred_entities = model.predict_entities(text, labels_list)

    true_ents = entity_set_from_bio(tokens, labels)
    pred_ents = entity_set_from_pred(pred_entities)

    # Union of labels from both sets (for false negatives, false positives)
    all_labels = set([ent[1] for ent in true_ents]) | set([ent[1] for ent in pred_ents])

    # For each true entity, label it as TP or FN
    for ent in true_ents:
        if ent in pred_ents:
            all_true_labels.append(ent[1])
            all_pred_labels.append(ent[1])
        else:
            all_true_labels.append(ent[1])
            all_pred_labels.append("O")  # False negative, predicted nothing

    # For predicted entities not in true, label as FP
    for ent in pred_ents:
        if ent not in true_ents:
            all_true_labels.append("O")  # True is nothing
            all_pred_labels.append(ent[1])

# Generate classification report
report = classification_report(all_true_labels, all_pred_labels, labels=labels_list, zero_division=0)
print(report)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


                          precision    recall  f1-score   support

           EquipmentName       0.69      0.10      0.17      1996
        DeviceProperties       0.09      0.02      0.03      2000
               FaultCode       0.66      0.62      0.64      2000
               FaultType       0.58      0.54      0.56      2000
       MaintenanceMethod       0.16      0.18      0.17      2000
VehicleComponentLocation       0.17      0.17      0.17      2000
        MeasurementValue       0.15      0.15      0.15      2000
        VehicleMakeModel       0.00      0.00      0.00      2000
            TimeDuration       0.15      0.17      0.16      2000
 CustomerReportedSymptom       0.36      0.01      0.03      2000

               micro avg       0.26      0.20      0.22     19996
               macro avg       0.30      0.20      0.21     19996
            weighted avg       0.30      0.20      0.21     19996



In [ ]:
#advarsarial

In [5]:
conll_filepath = "advarsarial.conll"

all_true_labels = []
all_pred_labels = []

# Entity list for prediction
labels_list = [
    "EquipmentName",
    "DeviceProperties",
    "FaultCode",
    "FaultType",
    "MaintenanceMethod",
    "VehicleComponentLocation",
    "MeasurementValue",
    "VehicleMakeModel",
    "TimeDuration",
    "CustomerReportedSymptom"
]

for tokens, labels in read_conll_file(conll_filepath):
    text = " ".join(tokens)
    pred_entities = model.predict_entities(text, labels_list)

    true_ents = entity_set_from_bio(tokens, labels)
    pred_ents = entity_set_from_pred(pred_entities)

    # Union of labels from both sets (for false negatives, false positives)
    all_labels = set([ent[1] for ent in true_ents]) | set([ent[1] for ent in pred_ents])

    # For each true entity, label it as TP or FN
    for ent in true_ents:
        if ent in pred_ents:
            all_true_labels.append(ent[1])
            all_pred_labels.append(ent[1])
        else:
            all_true_labels.append(ent[1])
            all_pred_labels.append("O")  # False negative, predicted nothing

    # For predicted entities not in true, label as FP
    for ent in pred_ents:
        if ent not in true_ents:
            all_true_labels.append("O")  # True is nothing
            all_pred_labels.append(ent[1])

# Generate classification report
report = classification_report(all_true_labels, all_pred_labels, labels=labels_list, zero_division=0)
print(report)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


                          precision    recall  f1-score   support

           EquipmentName       0.00      0.00      0.00       0.0
        DeviceProperties       0.00      0.00      0.00       0.0
               FaultCode       0.00      0.00      0.00       0.0
               FaultType       0.00      0.00      0.00       0.0
       MaintenanceMethod       0.00      0.00      0.00       0.0
VehicleComponentLocation       0.00      0.00      0.00       0.0
        MeasurementValue       0.00      0.00      0.00       0.0
        VehicleMakeModel       0.00      0.00      0.00       0.0
            TimeDuration       0.00      0.00      0.00       0.0
 CustomerReportedSymptom       0.00      0.00      0.00       0.0

               micro avg       0.00      0.00      0.00       0.0
               macro avg       0.00      0.00      0.00       0.0
            weighted avg       0.00      0.00      0.00       0.0



In [ ]:
conll_filepath = "advarsarial.conll"

all_true_labels = []
all_pred_labels = []

# Entity list for prediction
labels_list = [
    "EquipmentName",
    "DeviceProperties",
    "FaultCode",
    "FaultType",
    "MaintenanceMethod",
    "VehicleComponentLocation",
    "MeasurementValue",
    "VehicleMakeModel",
    "TimeDuration",
    "CustomerReportedSymptom"
]

for tokens, labels in read_conll_file(conll_filepath):
    text = " ".join(tokens)
    pred_entities = model.predict_entities(text, labels_list)

    true_ents = entity_set_from_bio(tokens, labels)
    pred_ents = entity_set_from_pred(pred_entities)

    # Union of labels from both sets (for false negatives, false positives)
    all_labels = set([ent[1] for ent in true_ents]) | set([ent[1] for ent in pred_ents])

    # For each true entity, label it as TP or FN
    for ent in true_ents:
        if ent in pred_ents:
            all_true_labels.append(ent[1])
            all_pred_labels.append(ent[1])
        else:
            all_true_labels.append(ent[1])
            all_pred_labels.append("O")  # False negative, predicted nothing

    # For predicted entities not in true, label as FP
    for ent in pred_ents:
        if ent not in true_ents:
            all_true_labels.append("O")  # True is nothing
            all_pred_labels.append(ent[1])

# Generate classification report
report = classification_report(all_true_labels, all_pred_labels, labels=labels_list, zero_division=0)
print(report)


In [7]:
#goldset

In [9]:
conll_filepath = "goldset.conll"

all_true_labels = []
all_pred_labels = []

# Entity list for prediction
labels_list = [
    "EquipmentName",
    "DeviceProperties",
    "FaultCode",
    "FaultType",
    "MaintenanceMethod",
    "VehicleComponentLocation",
    "MeasurementValue",
    "VehicleMakeModel",
    "TimeDuration",
    "CustomerReportedSymptom"
]

for tokens, labels in read_conll_file(conll_filepath):
    text = " ".join(tokens)
    pred_entities = model.predict_entities(text, labels_list)

    true_ents = entity_set_from_bio(tokens, labels)
    pred_ents = entity_set_from_pred(pred_entities)

    # Union of labels from both sets (for false negatives, false positives)
    all_labels = set([ent[1] for ent in true_ents]) | set([ent[1] for ent in pred_ents])

    # For each true entity, label it as TP or FN
    for ent in true_ents:
        if ent in pred_ents:
            all_true_labels.append(ent[1])
            all_pred_labels.append(ent[1])
        else:
            all_true_labels.append(ent[1])
            all_pred_labels.append("O")  # False negative, predicted nothing

    # For predicted entities not in true, label as FP
    for ent in pred_ents:
        if ent not in true_ents:
            all_true_labels.append("O")  # True is nothing
            all_pred_labels.append(ent[1])

# Generate classification report
report = classification_report(all_true_labels, all_pred_labels, labels=labels_list, zero_division=0)
print(report)


                          precision    recall  f1-score   support

           EquipmentName       0.00      0.00      0.00       0.0
        DeviceProperties       0.00      0.00      0.00       0.0
               FaultCode       0.00      0.00      0.00       0.0
               FaultType       0.00      0.00      0.00       0.0
       MaintenanceMethod       0.00      0.00      0.00       0.0
VehicleComponentLocation       0.00      0.00      0.00       0.0
        MeasurementValue       0.00      0.00      0.00       0.0
        VehicleMakeModel       0.00      0.00      0.00       0.0
            TimeDuration       0.00      0.00      0.00       0.0
 CustomerReportedSymptom       0.00      0.00      0.00       0.0

               micro avg       0.00      0.00      0.00       0.0
               macro avg       0.00      0.00      0.00       0.0
            weighted avg       0.00      0.00      0.00       0.0



In [13]:
from gliner import GLiNER
from sklearn.metrics import classification_report

# -------------------------------
# Helper Functions
# -------------------------------

def read_conll_file(filepath):
    """Read CoNLL file and yield tokens and BIO labels per sentence."""
    tokens, labels = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:  # sentence separator
                if tokens:
                    yield tokens, labels
                    tokens, labels = [], []
            else:
                splits = line.split()
                tokens.append(splits[0])
                labels.append(splits[1])
        if tokens:
            yield tokens, labels

def bio_to_entities(tokens, bio_labels):
    """Convert BIO labels to entity spans with start/end indices."""
    entities = []
    entity = None
    for i, tag in enumerate(bio_labels):
        if tag.startswith("B-"):
            if entity:
                entities.append(entity)
            entity = {"label": tag[2:], "tokens": [tokens[i]], "start": i, "end": i}
        elif tag.startswith("I-") and entity and tag[2:] == entity["label"]:
            entity["tokens"].append(tokens[i])
            entity["end"] = i
        else:
            if entity:
                entities.append(entity)
                entity = None
    if entity:
        entities.append(entity)
    return entities

def entity_set_from_bio(tokens, bio_labels):
    """Return set of (text, label) from BIO labels."""
    ents = bio_to_entities(tokens, bio_labels)
    return set((" ".join(e["tokens"]), e["label"]) for e in ents)

def entity_set_from_pred(pred_entities):
    """Convert GLiNER prediction output to set of (text, label)."""
    return set((e["text"], e["label"]) for e in pred_entities)

# -------------------------------
# Load GLiNER Model
# -------------------------------

model = GLiNER.from_pretrained("urchade/gliner_largev2")




Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [15]:
# Path to your CoNLL file
conll_filepath = "goldset.conll"

# List of entity types to evaluate
labels_list = [
    "EquipmentName",
    "DeviceProperties",
    "FaultCode",
    "FaultType",
    "MaintenanceMethod",
    "VehicleComponentLocation",
    "MeasurementValue",
    "VehicleMakeModel",
    "TimeDuration",
    "CustomerReportedSymptom"
]

all_true_labels = []
all_pred_labels = []

for tokens, labels in read_conll_file(conll_filepath):
    text = " ".join(tokens)
    
    # Get GLiNER predictions
    pred_entities = model.predict_entities(text, labels_list)
    
    # Convert BIO labels to entity sets
    true_ents = entity_set_from_bio(tokens, labels)
    pred_ents = entity_set_from_pred(pred_entities)
    
    # True positives and false negatives
    for ent in true_ents:
        if ent in pred_ents:
            all_true_labels.append(ent[1])
            all_pred_labels.append(ent[1])
        else:
            all_true_labels.append(ent[1])
            all_pred_labels.append("O")  # False negative

    # False positives
    for ent in pred_ents:
        if ent not in true_ents:
            all_true_labels.append("O")
            all_pred_labels.append(ent[1])

# -------------------------------
# Classification Report
# -------------------------------

report = classification_report(
    all_true_labels,
    all_pred_labels,
    labels=labels_list,
    zero_division=0
)

print(report)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


                          precision    recall  f1-score   support

           EquipmentName       0.00      0.00      0.00       0.0
        DeviceProperties       0.00      0.00      0.00       0.0
               FaultCode       0.00      0.00      0.00       0.0
               FaultType       0.00      0.00      0.00       0.0
       MaintenanceMethod       0.00      0.00      0.00       0.0
VehicleComponentLocation       0.00      0.00      0.00       0.0
        MeasurementValue       0.00      0.00      0.00       0.0
        VehicleMakeModel       0.00      0.00      0.00       0.0
            TimeDuration       0.00      0.00      0.00       0.0
 CustomerReportedSymptom       0.00      0.00      0.00       0.0

               micro avg       0.00      0.00      0.00       0.0
               macro avg       0.00      0.00      0.00       0.0
            weighted avg       0.00      0.00      0.00       0.0



In [21]:
def read_conll_labels(filepath):
    """
    Reads a CoNLL file and returns all true labels (BIO tags stripped to entity type).
    Works for 4-column files like your example.
    """
    true_labels = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("-DOCSTART-"):
                continue  # skip sentence separators and doc start
            parts = line.split()
            if len(parts) < 4:
                continue
            label = parts[3]  # BIO label is in 4th column
            if label != "O":
                true_labels.append(label[2:] if "-" in label else label)  # strip B-/I-
            else:
                true_labels.append("O")
    return true_labels

# -------------------------------
# Example usage
# -------------------------------
conll_file = "goldset.conll"
labels = read_conll_labels(conll_file)

print("Unique true labels in the file:")
print(set(labels))
print("Total labels:", len(labels))


Unique true labels in the file:
{'MeasurementValue', 'VehicleComponentLocation', 'DeviceProperties', 'EquipmentName', 'VehicleMakeModel', 'MaintenanceMethod', 'CustomerReportedSymptom', 'FaultCode', 'O', 'FaultType', 'TimeDuration'}
Total labels: 3513
